# Example notebook for an Fst statistic analysis

The present notebook serves as a guide of how to use the `IDEAL-GENOM-QC` library to compute Fst (fixation index) statistics between the study population and each 1000 Genomes super-population.

The underlying module (`ideal_genom.population.fst_stats`) provides the `FstSummary` class. It reuses `ReferenceGenomicMerger` (the same class used by `AncestryQC`) to harmonize and merge the study data with the 1000 Genomes reference panel, tags every sample with a `SuperPop` (or `StPop` for study samples without a reference match), and then runs **PLINK1.9**'s `--fst --within` for each reference super-population against the study population.

Let us import the required libraries.

In [ ]:
import sys
import os

import pandas as pd

from pathlib import Path

# add parent directory to path
library_path = os.path.abspath('..')
if library_path not in sys.path:
    sys.path.append(library_path)

library_path = Path(library_path)

from ideal_genom.population.fst_stats import FstSummary

In the next cell the path variables associated with the project are set.

As with ancestry QC and dimensionality reduction, this step is typically run on the cleaned output of the sample QC pipeline. Since each user can have a slightly different choice for the LD regions, the user can provide its own file; otherwise it is fetched automatically for builds **GRCh37** and **GRCh38**.

The user can also provide the path to the reference genome files (`reference_files` dict with `bed`/`bim`/`fam`/`psam` keys) or let the library fetch the 1000 Genomes data automatically (the default, used here).

In [ ]:
DATA_PATH = library_path / 'ideal_genom' / 'data'

test_data = DATA_PATH / 'test_data'
ouputData = test_data / 'outputData'

# Use the cleaned output of the sample QC notebook as input
input_path = ouputData / 'sample_qc_results' / 'clean_files'
input_name = '1KG_GRCh38_sample_qc'
output_path = ouputData
high_ld_file = Path('path/to/ld-regions/file') # if not available, set to Path()

Initialize the class `FstSummary`. Since no `reference_files` are provided, the 1000 Genomes reference panel will be fetched automatically for the chosen build.

If `recompute_merge` is `False`, the merging step below will be skipped and the merged `PLINK` files are expected to already exist under `fst_summary.merging_dir` (e.g. from a previous run).

In [ ]:
fst_summary = FstSummary(
    input_path     =input_path,
    input_name     =input_name,
    output_path    =output_path,
    high_ld_file   =high_ld_file,
    build          ='38', # '38' it is the default value
    recompute_merge=True, # if True, it will recompute the merge of the input files
)

`merge_reference_study()` merges the study data with the 1000 Genomes reference panel using the same harmonization steps as `AncestryQC`: renaming SNP IDs, filtering strand-ambiguous SNPs, LD pruning, fixing chromosome/position/allele-flip mismatches, and finally merging. `ind_pair` are the `--indep-pairwise` parameters used for LD pruning.

This step shells out to PLINK many times, which prints a lot of console text; we capture it into `fst_merge_log` to keep the notebook readable — run `fst_merge_log.show()` in a new cell if you need to inspect it.

In [ ]:
%%capture fst_merge_log
fst_summary.merge_reference_study(ind_pair=[50, 5, 0.2])

In [ ]:
print(f"Merging completed. Merged PLINK files written to: {fst_summary.merging_dir}")

`add_population_tags()` reads the reference panel's `PSAM` file to tag every reference sample with its `SuperPop`, and labels every study sample not present in the reference panel as `'StPop'`. The result is written to `fst_summary.population_tags`.

In [ ]:
fst_summary.add_population_tags()

In [ ]:
population_tags = pd.read_csv(fst_summary.population_tags, sep='\t')
population_tags['SuperPop'].value_counts()

`compute_fst()` reads the population tags and, for each reference `SuperPop` (excluding `'StPop'`), builds `--keep`/`--within` filter files and runs PLINK1.9's `--fst --within` to compare that super-population against the study population.

This also shells out to PLINK repeatedly; we capture its console output into `fst_compute_log` — run `fst_compute_log.show()` in a new cell if you need to inspect it.

In [ ]:
%%capture fst_compute_log
fst_summary.compute_fst()

In [ ]:
print(f"Fst computation completed. Results written to: {fst_summary.results_dir}")

`report_fst()` parses the `Mean Fst`/`Weighted Fst` lines out of each `fst-{SuperPop}-StPop.log` file, assembles them into a summary `DataFrame`, writes it to `fst_summary.csv` in the results directory, and removes the leftover intermediate `PLINK` binaries (`.bed`/`.bim`/`.fam`) from the results directory.

In [ ]:
fst_report = fst_summary.report_fst()
fst_report

The same summary is available on disk, in case you want to load it again without re-running the pipeline.

**Note:** unlike sample/variant/ancestry QC, there is no dedicated clean-up class for this module — the `keep-*`/`within-*` filter files and PLINK `.log` files are left in `fst_summary.results_dir` for inspection.

In [ ]:
fst_csv_path = fst_summary.results_dir / 'fst_summary.csv'
print(f"Fst summary report: {fst_csv_path} (exists: {fst_csv_path.exists()})")